In [0]:
import mlflow

model_uri = (
    "models:/mlops_demo.iris.iris_classifier/1"
)

model = mlflow.pyfunc.load_model(
    model_uri
)

In [0]:
predictions = model.predict(X_test)

In [0]:
prediction_df = X_test.copy()

prediction_df["actual"] = y_test.values
prediction_df["prediction"] = predictions

In [0]:
spark_prediction_df = spark.createDataFrame(
    prediction_df
)

spark_prediction_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "mlops_demo.iris.predictions"
    )

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    min,
    max
)

df = spark.table(
    "mlops_demo.iris.predictions"
)

df.select(
    count("*").alias("row_count"),
    avg("petal length (cm)").alias("avg_petal_length"),
    min("petal length (cm)").alias("min_petal_length"),
    max("petal length (cm)").alias("max_petal_length")
).display()